In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier


In [2]:
dataset1 = pd.read_csv("prep.csv")
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)
indep_x = df2.drop('classification_yes', axis=1)
dep_y = df2['classification_yes']

This Function performs Recursive Feature Elimination (RFE) using four different classification models and returns the selected features for each model.
1.Logistic Regression
2.SVM Linear
3.Random Forest and
4.Decision Tree

This function accepts three inputs - independent, dependent and number of features you want to select

rfelist will store the selected feature datasets produced by RFE for each model. After processing four models, it will contain four transformed
datasets.

Create the Classification models:
1.LogisticRegression
2.SVM Linear
3.Random Forest and
4.Decision Tree

Put all the models into a list named rfemodellist
Loop through each model
And find 5 best features for that model during fit method looking at the entire dataset
During Transform method, all the unwanted features are dropped and only the features selected by RFE is created in a new dataset

rfelist is appended with the best 5 features of each classification model during the loop.
This is the list containing 5 selected features of each classification model

rfelist[0] - LogisticRegression selected features
rfelist[1] - SVM Linear selected features
rfelist[2] - RandomForest selected features
rfelist[3] - DecisionTree selected features

The returned rfelist will contain the above information


In [3]:
def rfeFeature(indep_x, dep_y, n):
    rfelist = []
    log_model = LogisticRegression(solver='liblinear')
    RF = RandomForestClassifier(n_estimators=10, criterion = 'entropy', random_state=0)
    DT = DecisionTreeClassifier(criterion='gini', max_features='sqrt', splitter='best', random_state=0)
    svc_model = SVC(kernel='linear',random_state=0)
    rfemodellist = [log_model, svc_model, RF, DT]
    for i in rfemodellist:
        print(i)
        log_rfe = RFE(estimator=i,n_features_to_select=n)
        log_fit = log_rfe.fit(indep_x, dep_y)
        log_rfe_feature = log_fit.transform(indep_x)
        rfelist.append(log_rfe_feature)
    return rfelist
                

def split_scalar(indep_x, dep_y):
    x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.25, random_state=0)
    sc = StandardScaler()
    x_train = sc.fit_transform(x_train)
    x_test = sc.transform(x_test)
    return x_train, x_test, y_train, y_test
    

In [4]:
def cm_prediction(Classifier, x_test, y_test):
    from sklearn.metrics import confusion_matrix
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    y_pred = Classifier.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return Classifier, Accuracy, report, x_test, y_test, cm


In [5]:
#Logistic
def logistic(x_train, y_train, x_test, y_test):
    from sklearn.linear_model import LogisticRegression

    Classifier = LogisticRegression(random_state=0)

    Classifier.fit(x_train, y_train)

    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(
        Classifier, x_test, y_test
    )

    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMLinear
def svm_linear(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'linear', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMNonLinear
def svm_NL(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'rbf', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#Navie
def Navie(x_train, y_train, x_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    Classifier = GaussianNB()
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#KNN
def knn(x_train, y_train, x_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    Classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p=2)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#Decision
def Decision(x_train, y_train, x_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    Classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#RandomForest
def random(x_train, y_train, x_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    Classifier = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

In [6]:
def rfe_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf):
    rfedataframe = pd.DataFrame(index= ['Logistic', 'svc', 'Random','DecisionTree'], 
    columns=['Logistic', 'SVML', 'SVMNL', 'KNN', 'Navie', 'Decision', 'Random'])

    for number, idex in enumerate(rfedataframe.index):
        rfedataframe.loc[idex, 'Logistic']=acclog[number]
        rfedataframe.loc[idex, 'SVML'] = accsvml[number]
        rfedataframe.loc[idex,'SVMNL'] = accsvmnl[number]
        rfedataframe.loc[idex, 'KNN'] = accknn[number]
        rfedataframe.loc[idex, 'Navie'] = accnav[number]
        rfedataframe.loc[idex, 'Decision'] = accdes[number]
        rfedataframe.loc[idex, 'Random'] = accrf[number]
    return rfedataframe

In [7]:
rfelist = rfeFeature(indep_x, dep_y, 7)


LogisticRegression(solver='liblinear')
SVC(kernel='linear', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)


In [8]:
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []

for i in rfelist:
    x_train, x_test, y_train, y_test = split_scalar(i, dep_y)

    Classifier, Accuracy, report, x_test, y_test, cm = logistic(x_train, y_train, x_test, y_test)
    acclog.append(Accuracy)

    Classifier, Accuracy, report, x_test, y_test, cm = svm_linear(x_train, y_train, x_test, y_test)
    accsvml.append(Accuracy)
    
    Classifier, Accuracy, report, x_test, y_test, cm = svm_NL(x_train, y_train, x_test, y_test)
    accsvmnl.append(Accuracy)
    
    Classifier, Accuracy, report, x_test, y_test, cm = knn(x_train, y_train, x_test, y_test)
    accknn.append(Accuracy)
    
    Classifier, Accuracy, report, x_test, y_test, cm = Navie(x_train, y_train, x_test, y_test)
    accnav.append(Accuracy)
    
    Classifier, Accuracy, report, x_test, y_test, cm = Decision(x_train, y_train, x_test, y_test)
    accdes.append(Accuracy)
    
    Classifier, Accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
    accrf.append(Accuracy)



In [9]:
print(acclog)
print(accsvml)
print(accsvmnl)
print(accknn)
print(accnav)
print(accdes)
print(accrf)

[0.98, 0.99, 0.98, 0.95]
[0.98, 0.99, 0.98, 0.96]
[0.98, 0.99, 0.98, 0.96]
[0.97, 0.99, 0.98, 0.94]
[0.98, 0.99, 0.91, 0.88]
[0.98, 1.0, 0.96, 0.97]
[0.98, 0.99, 0.96, 0.95]


In [10]:
result_rfe_df =rfe_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf)

# Comparison of each Model's Accuracy values

In [11]:
result_rfe_df

,Logistic,SVML,SVMNL,KNN,Navie,Decision,Random
Logistic,0.98,0.98,0.98,0.97,0.98,0.98,0.98
svc,0.99,0.99,0.99,0.99,0.99,1.0,0.99
Random,0.98,0.98,0.98,0.98,0.91,0.96,0.96
DecisionTree,0.95,0.96,0.96,0.94,0.88,0.97,0.95


# Columns Selected for k=7